In [ ]:
from google.colab import files
uploaded = files.upload()

Saving BONE FASTA-20260307T160322Z-1-001.zip to BONE FASTA-20260307T160322Z-1-001 (1).zip


In [ ]:
!unzip "BONE FASTA-20260307T160322Z-1-001 (1).zip"

Archive:  BONE FASTA-20260307T160322Z-1-001 (1).zip
  inflating: BONE FASTA/P02467.fasta.txt  
  inflating: BONE FASTA/A6J7X7.fasta.txt  
  inflating: BONE FASTA/A0A8L2Q4D5.fasta.txt  
  inflating: BONE FASTA/P02454.fasta.txt  
  inflating: BONE FASTA/C0HJN5.fasta.txt  
  inflating: BONE FASTA/A0AAA9S8B5.fasta.txt  
  inflating: BONE FASTA/D3Z7D5.fasta.txt  
  inflating: BONE FASTA/C0HJN4.fasta.txt  
  inflating: BONE FASTA/F1MTP1.fasta.txt  
  inflating: BONE FASTA/P02457.fasta.txt  
  inflating: BONE FASTA/A0A8D1S2R2.fasta.txt  
  inflating: BONE FASTA/A0A8C4TW42.fasta.txt  
  inflating: BONE FASTA/F8WHV4.fasta.txt  
  inflating: BONE FASTA/A0A8C0QNV2.fasta.txt  
  inflating: BONE FASTA/A0A8C0Q9N3.fasta.txt  
  inflating: BONE FASTA/Q28668.fasta.txt  
  inflating: BONE FASTA/O46392.fasta.txt  
  inflating: BONE FASTA/Q5QNQ9.fasta.txt  
  inflating: BONE FASTA/P02456.fasta.txt  
  inflating: BONE FASTA/P02465.fasta.txt  
  inflating: BONE FASTA/A0A8D2BL93.fasta.txt  
  inflating: BONE

In [ ]:
!cat "BONE FASTA"/*.txt > combined_proteins.fasta

In [ ]:
FASTA_PATH = "combined_proteins.fasta"

In [ ]:
!head -3 combined_proteins.fasta
!grep -c "^>" combined_proteins.fasta

>tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN Collagen alpha-1(XXVII) chain OS=Bos taurus OX=9913 GN=COL27A1 PE=4 SV=2
MGAGSARGTRGTAAAAAAQGGGFLFTWILVSLTCHLASTQGAPEDVDVLQQLGLSWTKAV
GGRSPPPPGVIPFQTGFIFTQRARLQAPTAAVLPASLGTELALVLSLCSHRVNHAFLFAV
45


In [ ]:
!pip -q install biopython pandas

import os
import pandas as pd
from Bio import SeqIO

def cleavage_sites_trypsin(seq: str, proline_exception: bool = True):
    cuts = [0]
    for i, aa in enumerate(seq[:-1]):
        if aa in ("K", "R"):
            if proline_exception and seq[i+1] == "P":
                continue
            cuts.append(i+1)
    cuts.append(len(seq))
    return sorted(set(cuts))

def digest_protein(seq: str, missed_cleavages=2, min_len=7, max_len=35, proline_exception=True):
    seq = str(seq).replace("*", "").upper()
    cuts = cleavage_sites_trypsin(seq, proline_exception=proline_exception)
    peptides = []
    for start_i in range(len(cuts)-1):
        for mc in range(missed_cleavages + 1):
            end_i = start_i + 1 + mc
            if end_i >= len(cuts):
                continue
            start = cuts[start_i]
            end = cuts[end_i]
            pep = seq[start:end]
            if min_len <= len(pep) <= max_len:
                peptides.append(pep)
    return peptides

FASTA_PATH = "combined_proteins.fasta"

if not os.path.exists(FASTA_PATH):
    raise FileNotFoundError(f"{FASTA_PATH} not found")

records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print("Loaded proteins:", len(records))

if len(records) == 0:
    raise ValueError("No FASTA records found. Check combined_proteins.fasta content.")

rows = []
for rec in records:
    peptides = digest_protein(rec.seq, missed_cleavages=2, min_len=7, max_len=35)
    for pep in peptides:
        rows.append({
            "protein_id": rec.id,
            "peptide": pep,
            "pep_len": len(pep)
        })

df = pd.DataFrame(rows).drop_duplicates()
print("Total peptides:", len(df))

if df.empty:
    raise ValueError("Peptide table is empty. Check digestion parameters or FASTA content.")

df.to_csv("step1_digestion_peptides.csv", index=False)
print("Saved: step1_digestion_peptides.csv")
print(df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.6 MB/s eta 0:00:00
Loaded proteins: 45
Total peptides: 11241
Saved: step1_digestion_peptides.csv
                       protein_id                     peptide  pep_len
0  tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN                     MGAGSAR        7
1  tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN                  MGAGSARGTR       10
2  tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN    AVGGRSPPPPGVIPFQTGFIFTQR       24
3  tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN  AVGGRSPPPPGVIPFQTGFIFTQRAR       26
4  tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN         SPPPPGVIPFQTGFIFTQR       19


In [ ]:
import re
import pandas as pd
from Bio import SeqIO
from collections import defaultdict

FASTA_PATH = "combined_proteins.fasta"

def parse_uniprot_species(desc: str):
    os_match = re.search(r'OS=([^=]+?)\sOX=', desc)
    ox_match = re.search(r'OX=(\d+)', desc)
    species = os_match.group(1).strip() if os_match else "UNKNOWN"
    taxid = ox_match.group(1) if ox_match else "NA"
    return species, taxid

prot2species = {}
for rec in SeqIO.parse(FASTA_PATH, "fasta"):
    species, taxid = parse_uniprot_species(rec.description)
    prot2species[rec.id] = (species, taxid)

print("Example mapping:", list(prot2species.items())[:3])

df = pd.read_csv("step1_digestion_peptides.csv")

df["species"] = df["protein_id"].map(lambda x: prot2species.get(x, ("UNKNOWN","NA"))[0])
df["taxid"]   = df["protein_id"].map(lambda x: prot2species.get(x, ("UNKNOWN","NA"))[1])

pep2species = df.groupby("peptide")["species"].agg(lambda s: sorted(set(s))).reset_index()
pep2species["n_species"] = pep2species["species"].apply(len)

pep2species.to_csv("step2_peptide_species_map.csv", index=False)
df.to_csv("step2_peptides_with_species.csv", index=False)

print("Unique peptides:", pep2species.shape[0])
print("Peptides found in exactly 1 species:", (pep2species["n_species"]==1).sum())
print("Saved: step2_peptide_species_map.csv, step2_peptides_with_species.csv")

Example mapping: [('sp|P39059|COFA1_HUMAN', ('Homo sapiens', '9606')), ('sp|Q8NFW1|COMA1_HUMAN', ('Homo sapiens', '9606')), ('sp|Q9UMD9|COHA1_HUMAN', ('Homo sapiens', '9606'))]
Unique peptides: 13840
Peptides found in exactly 1 species: 11762
Saved: step2_peptide_species_map.csv, step2_peptides_with_species.csv


In [ ]:
import re
import pandas as pd
from Bio import SeqIO

FASTA_PATH = "combined_proteins.fasta"

def parse_uniprot_species(desc: str):
    os_match = re.search(r'OS=([^=]+?)\sOX=', desc)
    ox_match = re.search(r'OX=(\d+)', desc)
    species = os_match.group(1).strip() if os_match else "UNKNOWN"
    taxid = ox_match.group(1) if ox_match else "NA"
    return species, taxid

prot2species = {}
for rec in SeqIO.parse(FASTA_PATH, "fasta"):
    species, taxid = parse_uniprot_species(rec.description)
    prot2species[rec.id] = {
        "species": species,
        "taxid": taxid
    }

print("Example mapping:", list(prot2species.items())[:3])

df = pd.read_csv("step1_digestion_peptides.csv")

df["species"] = df["protein_id"].map(lambda x: prot2species.get(x, {}).get("species", "UNKNOWN"))
df["taxid"]   = df["protein_id"].map(lambda x: prot2species.get(x, {}).get("taxid", "NA"))

print("Unmapped protein IDs:", (df["species"] == "UNKNOWN").sum())

pep2map = (
    df.groupby("peptide")
      .agg(
          species=("species", lambda s: sorted(set(s))),
          taxids=("taxid", lambda s: sorted(set(s))),
          n_species=("species", lambda s: len(set(s))),
          n_taxids=("taxid", lambda s: len(set(s)))
      )
      .reset_index()
)

pep2map["species_list"] = pep2map["species"].apply(lambda x: "; ".join(x))
pep2map["taxid_list"] = pep2map["taxids"].apply(lambda x: "; ".join(x))

pep2map.to_csv("step2_peptide_species_map.csv", index=False)
df.to_csv("step2_peptides_with_species.csv", index=False)

print("Unique peptides:", pep2map.shape[0])
print("Peptides found in exactly 1 species:", (pep2map["n_species"] == 1).sum())
print("Saved: step2_peptide_species_map.csv")
print("Saved: step2_peptides_with_species.csv")
print(pep2map.head())

Example mapping: [('tr|A0A3Q1MT85|A0A3Q1MT85_BOVIN', {'species': 'Bos taurus', 'taxid': '9913'}), ('tr|A0A8C0Q9N3|A0A8C0Q9N3_CANLF', {'species': 'Canis lupus familiaris', 'taxid': '9615'}), ('tr|A0A8C0QB93|A0A8C0QB93_CANLF', {'species': 'Canis lupus familiaris', 'taxid': '9615'})]
Unmapped protein IDs: 0
Unique peptides: 6374
Peptides found in exactly 1 species: 5172
Saved: step2_peptide_species_map.csv
Saved: step2_peptides_with_species.csv
                        peptide                   species   taxids  n_species  \
0                      AAAGGSAR  [Canis lupus familiaris]   [9615]          1   
1               AAAGGSARTPLPPAK  [Canis lupus familiaris]   [9615]          1   
2                       AAASGSR            [Mus musculus]  [10090]          1   
3  AAASGSRGPGELGAPGPGTVALAEQCAR            [Mus musculus]  [10090]          1   
4                       AAATGAR            [Homo sapiens]   [9606]          1   

   n_taxids            species_list taxid_list  
0         1  Canis

In [ ]:
import pandas as pd

pep_map = pd.read_csv("step2_peptide_species_map.csv")
pep_occ = pd.read_csv("step2_peptides_with_species.csv")

unique_peps = pep_map[pep_map["n_species"] == 1].copy()


def first_species(x):
    s = str(x)
    s = s.strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
        s = s.strip("'").strip('"')
    return s.split(",")[0].strip().strip("'").strip('"')

unique_peps["unique_species"] = unique_peps["species"].apply(first_species)

print("Species-unique peptides:", len(unique_peps))
unique_peps.head()

Species-unique peptides: 5172


,peptide,species,taxids,n_species,n_taxids,species_list,taxid_list,unique_species
0,AAAGGSAR,['Canis lupus familiaris'],['9615'],1,1,Canis lupus familiaris,9615,Canis lupus familiaris
1,AAAGGSARTPLPPAK,['Canis lupus familiaris'],['9615'],1,1,Canis lupus familiaris,9615,Canis lupus familiaris
2,AAASGSR,['Mus musculus'],['10090'],1,1,Mus musculus,10090,Mus musculus
3,AAASGSRGPGELGAPGPGTVALAEQCAR,['Mus musculus'],['10090'],1,1,Mus musculus,10090,Mus musculus
4,AAATGAR,['Homo sapiens'],['9606'],1,1,Homo sapiens,9606,Homo sapiens


In [ ]:
import re
import pandas as pd
from Bio import SeqIO

FASTA_PATH = "combined_proteins.fasta"

def extract_pe(description):
    match = re.search(r"\bPE=(\d)\b", description)
    return int(match.group(1)) if match else None

protein_pe = {}
protein_existence_score = {}

for record in SeqIO.parse(FASTA_PATH, "fasta"):
    pe = extract_pe(record.description)
    protein_pe[record.id] = pe
    protein_existence_score[record.id] = (6 - pe) if pe else None

print("Parsed PE for", len(protein_pe), "proteins")


pep_occ = pd.read_csv("step2_peptides_with_species.csv")

def clean_species(x):
    x = str(x)
    if x.startswith("[") and x.endswith("]"):
        x = x[1:-1]
    return x.strip().strip("'").strip('"')

pep_occ["species"] = pep_occ["species"].apply(clean_species)

pep_occ["protein_existence_score"] = pep_occ["protein_id"].map(protein_existence_score)


pep_map = pd.read_csv("step2_peptide_species_map.csv")

unique_peptides = pep_map[pep_map["n_species"] == 1].copy()

unique_set = set(unique_peptides["peptide"])

unique_occ = pep_occ[pep_occ["peptide"].isin(unique_set)].copy()

peptide_scores = (
    unique_occ.groupby(["peptide", "species"])
    .agg(
        peptide_score=("protein_existence_score", "max"),
        n_proteins=("protein_id", "nunique")
    )
    .reset_index()
)

peptide_scores.to_csv("step3_unique_peptides_scored.csv", index=False)

print("Saved: step3_unique_peptides_scored.csv")


species_summary = (
    peptide_scores.groupby("species")
    .agg(
        unique_peptide_count=("peptide", "nunique"),
        sum_peptide_score=("peptide_score", "sum"),
        mean_peptide_score=("peptide_score", "mean")
    )
    .reset_index()
    .sort_values("unique_peptide_count", ascending=False)
)

species_summary.to_csv("step3_species_summary.csv", index=False)

print("Saved: step3_species_summary.csv")


protein_table = pd.DataFrame({
    "protein_id": list(protein_pe.keys()),
    "PE": [protein_pe[p] for p in protein_pe],
    "protein_existence_score": [protein_existence_score[p] for p in protein_pe]
})

protein_table.to_csv("step3_protein_existence_scores.csv", index=False)

print("Saved: step3_protein_existence_scores.csv")

print("\nSpecies Summary Preview:")
print(species_summary.head())

Parsed PE for 45 proteins
Saved: step3_unique_peptides_scored.csv
Saved: step3_species_summary.csv
Saved: step3_protein_existence_scores.csv

Species Summary Preview:
             species  unique_peptide_count  sum_peptide_score  \
6       Mus musculus                  2057               9527   
5       Homo sapiens                  1052               5260   
0         Bos taurus                   435               1707   
3      Gallus gallus                   423               2021   
9  Rattus norvegicus                   340               1187   

   mean_peptide_score  
6            4.631502  
5            5.000000  
0            3.924138  
3            4.777778  
9            3.491176  


In [ ]:
import pandas as pd

pep_map = pd.read_csv("step2_peptide_species_map.csv")
pep_occ = pd.read_csv("step2_peptides_with_species.csv")

unique_peptides = pep_map[pep_map["n_species"] == 1][["peptide"]]

unique_species_map = (
    unique_peptides
    .merge(pep_occ[["peptide","species","taxid"]].drop_duplicates(),
           on="peptide")
)

unique_counts = (
    unique_species_map
    .groupby(["species","taxid"])["peptide"]
    .nunique()
    .reset_index(name="number_unique_peptides")
    .sort_values("number_unique_peptides", ascending=False)
)

unique_counts

,species,taxid,number_unique_peptides
6,Mus musculus,10090,2057
5,Homo sapiens,9606,1052
0,Bos taurus,9913,435
3,Gallus gallus,9031,423
9,Rattus norvegicus,10116,340
1,Canis lupus familiaris,9615,253
4,Hippopotamus amphibius,9833,178
2,Falco tinnunculus,100819,156
10,Sus scrofa,9823,150
7,Orycteropus afer,9818,84


In [ ]:
import pandas as pd

pep_occ = pd.read_csv("step2_peptides_with_species.csv")

TOTAL_SPECIES = 16

protein_species = (
    pep_occ[["protein_id","species"]]
    .drop_duplicates()
    .groupby("protein_id")["species"]
    .nunique()
    .reset_index(name="n_species_protein")
)

protein_species["protein_existence_score"] = (
    protein_species["n_species_protein"] / TOTAL_SPECIES
)

protein_species.head()

,protein_id,n_species_protein,protein_existence_score
0,sp|A6H584|CO6A5_MOUSE,1,0.0625
1,sp|C0HJN4|CO1A2_ORYAF,1,0.0625
2,sp|C0HJN5|CO1A1_HIPAM,1,0.0625
3,sp|C0HJN6|CO1A2_HIPAM,1,0.0625
4,sp|O46392|CO1A2_CANLF,1,0.0625


In [ ]:
pep_map = pd.read_csv("step2_peptide_species_map.csv")

pep_map["peptide_existence_score"] = (
    pep_map["n_species"] / TOTAL_SPECIES
)

pep_map[["peptide","n_species","peptide_existence_score"]].head()

,peptide,n_species,peptide_existence_score
0,AAAGGSAR,1,0.0625
1,AAAGGSARTPLPPAK,1,0.0625
2,AAASGSR,1,0.0625
3,AAASGSRGPGELGAPGPGTVALAEQCAR,1,0.0625
4,AAATGAR,1,0.0625


In [ ]:

from google.colab import files

files.download("step3_unique_peptides_scored.csv")
files.download("step3_species_summary.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>